## Реализация градиентного спуска

Реализуйте линейную регрессию с функцией потерь MSE, обучаемую с помощью:

**Задание 1** Градиентного спуска;

**Задание 2** Стохастического градиентного спуска.


Во всех пунктах необходимо соблюдать следующие условия:

* Все вычисления должны быть векторизованы;
* Циклы средствами python допускается использовать только для итераций градиентного спуска;
* В качестве критерия останова необходимо использовать (одновременно):

    * проверку на евклидовую норму разности весов на двух соседних итерациях (например, меньше некоторого малого числа порядка $10^{-6}$, задаваемого параметром `tolerance`);
    * достижение максимального числа итераций (например, 10000, задаваемого параметром `max_iter`).
* Чтобы проследить, что оптимизационный процесс действительно сходится, будем использовать атрибут класса `loss_history` — в нём после вызова метода `fit` должны содержаться значения функции потерь для всех итераций, начиная с первой (до совершения первого шага по антиградиенту);
* Инициализировать веса можно случайным образом или нулевым вектором.


Ниже приведён шаблон класса, который должен содержать код реализации каждого из методов.

## Формула градиента

Функция потерь в векторном (матричном) виде можно представить следующим образом:

$$Q(w) = \frac{ 1 }{ n } \|X w - y\|^2$$


Тогда производная этой функции будет принимать следующий вид (производная вычисляется как **производная сложной функции**):

$$\frac{ dQ }{ d w }= \frac{ 1 }{ n } (\|X w - y\|^2)' * (X w - y)'  =  \frac{ 1 }{ n } 2(X w -y)X = \frac{ 1 }{ n } 2X(Xw-y)$$

Но, т. к. здесь используются матрицы, то $X$ перед скобками необходимо транспонировать, чтобы было возможным осуществить операцию векторного умножения, нужно соблюсти правило:
> Чтобы матрицу A можно было умножить на матрицу B нужно, чтобы число столбцов матрицы A равнялось числу строк матрицы B.

Поэтому, производная функции потерь будет иметь следующий вид:

$$\frac{ dQ }{ d w }= \frac{ 1 }{ n } 2X^{T}(Xw-y)$$

Т. к. в выражении 1 переменная, то градиент будет выглядеть так:

$$\nabla Q(w) = \frac{ 1 }{ n } 2X^{T}(Xw-y) $$

Подробнее см. раздел 5.3  Линейная регрессия в [Математические методы обучения по прецедентам (теория обучения машин)](http://www.machinelearning.ru/wiki/images/6/6d/Voron-ML-1.pdf)

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

class LinearReg(BaseEstimator):
    def __init__(self, gd_type='stochastic',
                 tolerance=1e-4, max_iter=1000, w0=None, alpha=1e-3, eta=1e-2):
        self.gd_type = gd_type
        self.tolerance = tolerance
        self.max_iter = max_iter
        self.w0 = w0
        self.alpha = alpha  # не используется, но оставим
        self.eta = eta
        self.w = None
        self.loss_history = None

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        
        self.loss_history = []
        
        # Инициализация весов
        if self.w0 is None:
            self.w = np.zeros(X.shape[1])
        else:
            self.w = self.w0.copy()

        # Первая потеря (до обучения)
        self.loss_history.append(self.calc_loss(X, y))

        for i in range(self.max_iter):
            w_prev = self.w.copy()

            if self.gd_type == 'full':
                grad = self.calc_gradient(X, y)
            else:  # stochastic
                idx = np.random.randint(0, X.shape[0])
                X_i = X[idx:idx+1]
                y_i = y[idx:idx+1]
                grad = self.calc_gradient(X_i, y_i)

            self.w = self.w - self.eta * grad

            # Критерий останова
            if np.linalg.norm(self.w - w_prev) < self.tolerance:
                print(f"Converged at iteration {i}")
                break

            # Сохраняем потерю после шага
            self.loss_history.append(self.calc_loss(X, y))

        return self

    def predict(self, X):
        if self.w is None:
            raise Exception('Not trained yet')
        X = np.array(X, dtype=float)
        return X.dot(self.w)

    def calc_gradient(self, X, y):
        # Универсальная формула: работает и для full, и для SGD
        return 2 * X.T.dot(X.dot(self.w) - y) / X.shape[0]

    def calc_loss(self, X, y):
        return np.mean((y - self.predict(X)) ** 2)

**Задание 3**.
* Загрузите данные из лабораторной работы № 3 ([train.csv](https://www.kaggle.com/c/nyc-taxi-trip-duration/data));
* Разбейте выборку на обучающую и тестовую в отношении 7:3 с random_seed=0;
* Преобразуйте целевую переменную `trip_duration` как $\hat{y} = \log{(y + 1)}$.

In [2]:
import pandas as pd
data = pd.read_csv('../lab2/train.csv')

In [ ]:
y = np.log1p(data['trip_duration'])
data = data.drop(columns=['pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'id', 'trip_duration'])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

/var/folders/1f/qzsxtjvd1hgcnxb8nykmnbz00000gn/T/ipykernel_26207/2420885091.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.11111111 0.11111111 0.11111111 ... 0.11111111 0.11111111 0.11111111]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[0:, :] = MinMaxScaler().fit_transform(X)


In [6]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train = X_train_scaled
X_test = X_test_scaled
y_train = y_train.values
y_test = y_test.values

**Задание 4**. Обучите и провалидируйте модели на данных из предыдущего пункта, сравните качество между методами по метрикам MSE и $R^2$. Исследуйте влияние параметров `max_iter` и `eta` на процесс оптимизации. Согласуется ли оно с вашими ожиданиями?

In [7]:
model_full = LinearReg(
    gd_type='full',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=1000,
    eta=1e-2
)
model_full.fit(X_train, y_train)

,gd_type,'full'
,tolerance,1e-06
,max_iter,1000
,w0,array([-0.021... 2.37839391])
,alpha,0.001
,eta,0.01


In [8]:
from sklearn.metrics import mean_squared_error, r2_score
y_pred_full_train = model_full.predict(X_train)
y_pred_full_test = model_full.predict(X_test)

In [9]:
mse_full_train = mean_squared_error(y_train, y_pred_full_train)
mse_full_test = mean_squared_error(y_test, y_pred_full_test)
r2_full_train = r2_score(y_train, y_pred_full_train)
r2_full_test = r2_score(y_test, y_pred_full_test)

In [10]:
model_sgd = LinearReg(
    gd_type='stochastic',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=1000,
    eta=1e-2
)
model_sgd.fit(X_train, y_train)

,gd_type,'stochastic'
,tolerance,1e-06
,max_iter,1000
,w0,array([0.2141... 2.69195343])
,alpha,0.001
,eta,0.01


In [11]:
y_pred_sgd_train = model_sgd.predict(X_train)
y_pred_sgd_test = model_sgd.predict(X_test)

In [12]:
mse_sgd_train = mean_squared_error(y_train, y_pred_sgd_train)
mse_sgd_test = mean_squared_error(y_test, y_pred_sgd_test)
r2_sgd_train = r2_score(y_train, y_pred_sgd_train)
r2_sgd_test = r2_score(y_test, y_pred_sgd_test)

In [13]:
results_comparison = pd.DataFrame({
    'Method': ['Full Gradient Descent', 'Stochastic Gradient Descent'],
    'Train MSE': [mse_full_train, mse_sgd_train],
    'Test MSE': [mse_full_test, mse_sgd_test],
    'Train R2': [r2_full_train, r2_sgd_train],
    'Test R2': [r2_full_test, r2_sgd_test],
})

print("Comparison of Validation Results:")
print(results_comparison)

Comparison of Validation Results:
                        Method  Train MSE  Test MSE  Train R2   Test R2
0        Full Gradient Descent   0.649116  0.644315 -0.022602 -0.023293
1  Stochastic Gradient Descent   0.694897  0.690365 -0.094724 -0.096430


In [14]:
model_full = LinearReg(
    gd_type='full',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=1000,
    eta=1e-3
)
model_full.fit(X_train, y_train)
y_pred_full_train = model_full.predict(X_train)
y_pred_full_test = model_full.predict(X_test)
mse_full_train = mean_squared_error(y_train, y_pred_full_train)
mse_full_test = mean_squared_error(y_test, y_pred_full_test)
r2_full_train = r2_score(y_train, y_pred_full_train)
r2_full_test = r2_score(y_test, y_pred_full_test)
model_sgd = LinearReg(
    gd_type='stochastic',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=1000,
    eta=1e-3
)
model_sgd.fit(X_train, y_train)
y_pred_sgd_train = model_sgd.predict(X_train)
y_pred_sgd_test = model_sgd.predict(X_test)
mse_sgd_train = mean_squared_error(y_train, y_pred_sgd_train)
mse_sgd_test = mean_squared_error(y_test, y_pred_sgd_test)
r2_sgd_train = r2_score(y_train, y_pred_sgd_train)
r2_sgd_test = r2_score(y_test, y_pred_sgd_test)
results_comparison = pd.DataFrame({
    'Method': ['Full Gradient Descent', 'Stochastic Gradient Descent'],
    'Train MSE': [mse_full_train, mse_sgd_train],
    'Test MSE': [mse_full_test, mse_sgd_test],
    'Train R2': [r2_full_train, r2_sgd_train],
    'Test R2': [r2_full_test, r2_sgd_test],
})

print("Comparison of Validation Results:")
print(results_comparison)

Comparison of Validation Results:
                        Method  Train MSE  Test MSE  Train R2   Test R2
0        Full Gradient Descent   0.988061  0.985842 -0.556568 -0.565703
1  Stochastic Gradient Descent   1.145434  1.143597 -0.804490 -0.816248


In [15]:
model_full = LinearReg(
    gd_type='full',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=1000,
    eta=1e-1
)
model_full.fit(X_train, y_train)
y_pred_full_train = model_full.predict(X_train)
y_pred_full_test = model_full.predict(X_test)
mse_full_train = mean_squared_error(y_train, y_pred_full_train)
mse_full_test = mean_squared_error(y_test, y_pred_full_test)
r2_full_train = r2_score(y_train, y_pred_full_train)
r2_full_test = r2_score(y_test, y_pred_full_test)
model_sgd = LinearReg(
    gd_type='stochastic',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=1000,
    eta=1e-1
)
model_sgd.fit(X_train, y_train)
y_pred_sgd_train = model_sgd.predict(X_train)
y_pred_sgd_test = model_sgd.predict(X_test)
mse_sgd_train = mean_squared_error(y_train, y_pred_sgd_train)
mse_sgd_test = mean_squared_error(y_test, y_pred_sgd_test)
r2_sgd_train = r2_score(y_train, y_pred_sgd_train)
r2_sgd_test = r2_score(y_test, y_pred_sgd_test)
results_comparison = pd.DataFrame({
    'Method': ['Full Gradient Descent', 'Stochastic Gradient Descent'],
    'Train MSE': [mse_full_train, mse_sgd_train],
    'Test MSE': [mse_full_test, mse_sgd_test],
    'Train R2': [r2_full_train, r2_sgd_train],
    'Test R2': [r2_full_test, r2_sgd_test],
})

print("Comparison of Validation Results:")
print(results_comparison)

Comparison of Validation Results:
                        Method  Train MSE  Test MSE  Train R2   Test R2
0        Full Gradient Descent   0.635825  0.630770 -0.001664 -0.001782
1  Stochastic Gradient Descent   0.950659  0.944908 -0.497646 -0.500691


In [16]:
model_full = LinearReg(
    gd_type='full',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=100,
    eta=1e-1
)
model_full.fit(X_train, y_train)
y_pred_full_train = model_full.predict(X_train)
y_pred_full_test = model_full.predict(X_test)
mse_full_train = mean_squared_error(y_train, y_pred_full_train)
mse_full_test = mean_squared_error(y_test, y_pred_full_test)
r2_full_train = r2_score(y_train, y_pred_full_train)
r2_full_test = r2_score(y_test, y_pred_full_test)
model_sgd = LinearReg(
    gd_type='stochastic',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=100,
    eta=1e-1
)
model_sgd.fit(X_train, y_train)
y_pred_sgd_train = model_sgd.predict(X_train)
y_pred_sgd_test = model_sgd.predict(X_test)
mse_sgd_train = mean_squared_error(y_train, y_pred_sgd_train)
mse_sgd_test = mean_squared_error(y_test, y_pred_sgd_test)
r2_sgd_train = r2_score(y_train, y_pred_sgd_train)
r2_sgd_test = r2_score(y_test, y_pred_sgd_test)
results_comparison = pd.DataFrame({
    'Method': ['Full Gradient Descent', 'Stochastic Gradient Descent'],
    'Train MSE': [mse_full_train, mse_sgd_train],
    'Test MSE': [mse_full_test, mse_sgd_test],
    'Train R2': [r2_full_train, r2_sgd_train],
    'Test R2': [r2_full_test, r2_sgd_test],
})

print("Comparison of Validation Results:")
print(results_comparison)

Comparison of Validation Results:
                        Method  Train MSE  Test MSE  Train R2   Test R2
0        Full Gradient Descent   0.644879  0.640040 -0.015927 -0.016504
1  Stochastic Gradient Descent   0.648233  0.643091 -0.021210 -0.021350


In [17]:
model_full = LinearReg(
    gd_type='full',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=500,
    eta=1e-1
)
model_full.fit(X_train, y_train)
y_pred_full_train = model_full.predict(X_train)
y_pred_full_test = model_full.predict(X_test)
mse_full_train = mean_squared_error(y_train, y_pred_full_train)
mse_full_test = mean_squared_error(y_test, y_pred_full_test)
r2_full_train = r2_score(y_train, y_pred_full_train)
r2_full_test = r2_score(y_test, y_pred_full_test)
model_sgd = LinearReg(
    gd_type='stochastic',
    w0=np.random.rand(X_train.shape[1]),
    tolerance=1e-6,
    max_iter=500,
    eta=1e-1
)
model_sgd.fit(X_train, y_train)
y_pred_sgd_train = model_sgd.predict(X_train)
y_pred_sgd_test = model_sgd.predict(X_test)
mse_sgd_train = mean_squared_error(y_train, y_pred_sgd_train)
mse_sgd_test = mean_squared_error(y_test, y_pred_sgd_test)
r2_sgd_train = r2_score(y_train, y_pred_sgd_train)
r2_sgd_test = r2_score(y_test, y_pred_sgd_test)
results_comparison = pd.DataFrame({
    'Method': ['Full Gradient Descent', 'Stochastic Gradient Descent'],
    'Train MSE': [mse_full_train, mse_sgd_train],
    'Test MSE': [mse_full_test, mse_sgd_test],
    'Train R2': [r2_full_train, r2_sgd_train],
    'Test R2': [r2_full_test, r2_sgd_test],
})

print("Comparison of Validation Results:")
print(results_comparison)

Comparison of Validation Results:
                        Method  Train MSE  Test MSE  Train R2   Test R2
0        Full Gradient Descent   0.636091  0.631064 -0.002083 -0.002249
1  Stochastic Gradient Descent   0.772915  0.768681 -0.217632 -0.220811


In [18]:
max_iter=1000
eta=1e-1
